# Vani-Kanoon Legal LLM Training (Fixed)

Fine-tune a small LLM for Indian legal assistance using LoRA.

## Setup:
1. Upload `training_data.json` to Kaggle
2. Enable GPU T4 x2
3. Enable Internet
4. Run all cells in order

In [ ]:
# Install specific versions to avoid compatibility issues
!pip install -q transformers==4.40.0 datasets accelerate peft bitsandbytes trl==0.8.6

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Training Data

In [ ]:
# Load training data - update path if needed
# If uploaded as dataset: "/kaggle/input/vani-kanoon-training/training_data.json"
data_path = "/kaggle/input/vani-kanoon-training/training_data.json"

try:
    with open(data_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
except:
    # Try alternate path
    data_path = "training_data.json"
    with open(data_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

print(f"Loaded {len(raw_data)} training examples")

In [ ]:
# Format data for training
def format_example(example):
    instruction = example.get('instruction', '')
    input_text = example.get('input', '')
    output = example.get('output', '')
    
    if input_text:
        text = f"""<s>[INST] {instruction}

Input: {input_text} [/INST] {output}</s>"""
    else:
        text = f"""<s>[INST] {instruction} [/INST] {output}</s>"""
    
    return {"text": text}

formatted_data = [format_example(ex) for ex in raw_data]
dataset = Dataset.from_list(formatted_data)

print(f"Dataset: {len(dataset)} examples")
print(f"\nSample:\n{dataset[0]['text'][:300]}...")

## 2. Load Model

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded")

In [ ]:
# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {model.num_parameters():,} parameters")

## 3. Configure LoRA

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 4. Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024,
        padding="max_length",
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

# Add labels (same as input_ids for causal LM)
def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

tokenized_dataset = tokenized_dataset.map(add_labels, batched=True)

print(f"Tokenized dataset ready: {len(tokenized_dataset)} examples")

## 5. Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./vani-kanoon-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Trainer ready!")

In [ ]:
# Start training
print("="*50)
print("TRAINING STARTED")
print("="*50)

trainer.train()

print("="*50)
print("TRAINING COMPLETED!")
print("="*50)

## 6. Save Model

In [ ]:
# Save LoRA adapter
SAVE_PATH = "./vani-kanoon-lora"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Model saved to: {SAVE_PATH}")

In [ ]:
# Create zip for download
import shutil
import os

shutil.make_archive("vani-kanoon-lora", 'zip', SAVE_PATH)
size_mb = os.path.getsize("vani-kanoon-lora.zip") / (1024*1024)
print(f"Created: vani-kanoon-lora.zip ({size_mb:.1f} MB)")

## 7. Test the Model

In [ ]:
def ask(question, max_tokens=300):
    prompt = f"<s>[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("[/INST]")[-1].strip()

In [ ]:
# Test questions
questions = [
    "What is the punishment for murder under BNS 2023?",
    "How to file an FIR in India?",
    "धारा 302 में क्या सजा है?",
]

for q in questions:
    print(f"\n{'='*50}")
    print(f"Q: {q}")
    print(f"{'='*50}")
    print(f"A: {ask(q)}")

## Done!

Download `vani-kanoon-lora.zip` from the Output section on the right.

### To use with Ollama:
```bash
# Create Modelfile
FROM mistral:7b-instruct
ADAPTER ./vani-kanoon-lora
PARAMETER temperature 0.7
SYSTEM "You are Vani-Kanoon, an expert Indian legal assistant."

# Build model
ollama create vani-kanoon -f Modelfile
```